# Stable Challenge baseline in Google Colab

This notebook runs the baseline track end to end on a Colab runtime:

1. Clone `challenge-finance-ssl`.
2. Install the project dependencies.
3. Download and extract the official challenge corpus.
4. Profile the corpus.
5. Train the baseline encoder.
6. Export and validate the ONNX submission.
7. Run the synthetic proxy evaluation.

The corpus is large. Use a Colab runtime with enough disk space and GPU enabled.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/harshitaggarwal01/ssl-time-series.git"
BRANCH = "challenge-finance-ssl"
WORKSPACE = Path("/content/ssl-time-series")
PROJECT = WORKSPACE / "ssl-time-series"
CORPUS_DIR = PROJECT / "corpus"
OUTPUT_DIR = PROJECT / "outputs"
CORPUS_ZIP = Path("/content/corpus.zip")
DRIVE_FILE_ID = "1ld4mifcgwaBZin7cGBNCYm7xXH2T90U6"
+
def run(command, cwd=None):
    print("$", command)
    subprocess.run(command, cwd=cwd, shell=True, check=True)
+
print("GPU:", shutil.which("nvidia-smi") is not None)
run("df -h /content")
if WORKSPACE.exists():
    run(f"rm -rf {WORKSPACE}")
run(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {WORKSPACE}")
run("git log -1 --oneline", cwd=PROJECT)


In [ ]:
# Install uv and the project environment.
run("python -m pip install -q uv gdown")
run("uv sync", cwd=PROJECT)
run("uv run python -c 'import torch, numpy, onnx, onnxruntime; print(\"dependencies ok\"); print(\"torch\", torch.__version__, \"cuda\", torch.cuda.is_available())'", cwd=PROJECT)


In [ ]:
# Download and extract the official corpus.
# The archive is approximately 8.9 GB; extraction produces corpus/corpus.npz.
if not (CORPUS_DIR / "corpus.npz").exists():
    run(f"gdown --id {DRIVE_FILE_ID} -O {CORPUS_ZIP}")
    run(f"mkdir -p {CORPUS_DIR} && unzip -q {CORPUS_ZIP} -d {CORPUS_DIR}")
    run(f"rm -f {CORPUS_ZIP}")
run(f"ls -lh {CORPUS_DIR}/corpus.npz")
run("df -h /content")


In [ ]:
# Profile the corpus without loading all values into RAM.
run("uv run python -m scripts.profile_corpus --corpus corpus --out outputs/corpus_profile.json", cwd=PROJECT)


## Baseline training

The default run uses the reproducible baseline configuration. Set `SMOKE = True` for a short pipeline check before committing to the full run.

In [ ]:
SMOKE = False
steps = 10 if SMOKE else 2000
checkpoint = "outputs/baseline-smoke.pt" if SMOKE else "outputs/baseline.pt"
run(
    f"uv run python -m scripts.train_encoder --corpus corpus --config configs/baseline.json "
    f"--steps {steps} --output {checkpoint}",
    cwd=PROJECT,
)


In [ ]:
# Export only the encoder to the required static-batch ONNX format.
model_name = "baseline-smoke.onnx" if SMOKE else "baseline.onnx"
run(
    f"uv run python -m scripts.export_model --checkpoint {checkpoint} --output outputs/{model_name}",
    cwd=PROJECT,
)
run(f"ls -lh outputs/{model_name}", cwd=PROJECT)


In [ ]:
# Run the official local submission checks.
run(f"uv run test_submission.py outputs/{model_name}", cwd=PROJECT)


In [ ]:
# Run the synthetic proxy-task evaluation.
run(f"uv run python -m scripts.proxy_evaluate outputs/{model_name}", cwd=PROJECT)


In [ ]:
# Collect a compact result summary for the team.
summary = {
    "branch": BRANCH,
    "commit": subprocess.check_output("git rev-parse HEAD", cwd=PROJECT, shell=True, text=True).strip(),
    "model": str(PROJECT / "outputs" / model_name),
    "model_bytes": (PROJECT / "outputs" / model_name).stat().st_size,
    "profile": str(PROJECT / "outputs" / "corpus_profile.json"),
    "smoke": SMOKE,
}
import json
summary_path = OUTPUT_DIR / ("baseline_smoke_summary.json" if SMOKE else "baseline_summary.json")
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))


## Optional: copy artifacts to Google Drive

Run the next cell only if you want to keep the checkpoint, ONNX model, profile, and summary after the Colab runtime ends.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# destination = Path('/content/drive/MyDrive/stable-challenge-baseline')
# destination.mkdir(parents=True, exist_ok=True)
# run(f"cp -r outputs {destination}", cwd=PROJECT)
# print(destination)
